# M7U4 — Data Quality Queries (HVAC dataset)

Runs the same eight data quality queries used on the Duplex Apartment, plus three MEP-specific extensions, against the **`hvac`** database containing the NBU MedicalClinic HVAC model.

Results are written to `results/hvac/` to keep them separate from the Duplex results.

**Pre-requisite:** the `hvac` database must already be loaded via `03_extract_and_load_hvac.ipynb`.

In [1]:
# Cell 1 — Setup
import os
from pathlib import Path
import pandas as pd
from neo4j import GraphDatabase
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "results" / "hvac"
QUERIES_DIR = PROJECT_ROOT / "queries" / "hvac"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
QUERIES_DIR.mkdir(parents=True, exist_ok=True)

DATABASE = "hvac"

load_dotenv(PROJECT_ROOT / ".env")
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
)

def run_query(cypher, **params):
    """Run a Cypher query against the hvac database and return a DataFrame."""
    with driver.session(database=DATABASE) as session:
        result = session.run(cypher, **params)
        df = pd.DataFrame([r.data() for r in result])
    return df

def save_query_and_results(q_id, q_name, cypher, df):
    cypher_path = QUERIES_DIR / f"{q_id}_{q_name}.cypher"
    csv_path    = RESULTS_DIR / f"{q_id}_{q_name}.csv"
    cypher_path.write_text(cypher.strip() + "\n", encoding="utf-8")
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"  Saved query -> queries/hvac/{cypher_path.name}")
    print(f"  Saved CSV   -> results/hvac/{csv_path.name} ({len(df)} rows)")

# Sanity check — confirm hvac database has data
with driver.session(database=DATABASE) as s:
    n = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    r = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
print(f"Target database: {DATABASE}")
print(f"  Nodes: {n}")
print(f"  Relationships: {r}")
print("\nReady to run queries.")

Target database: hvac
  Nodes: 193689
  Relationships: 200262

Ready to run queries.


## Q1 — Elements with no property sets

In [2]:
Q1_NAME = "elements_without_psets"
Q1_CYPHER = """
MATCH (e:Element)
WHERE NOT (e)-[:HAS_PSET]->()
RETURN e.GlobalId AS GlobalId,
       e.IfcClass AS IfcClass,
       e.Name     AS Name,
       e.Tag      AS Tag
ORDER BY e.IfcClass, e.Name
"""
df_q1 = run_query(Q1_CYPHER)
print(f"Q1: {len(df_q1)} elements have no property sets.")
save_query_and_results("Q1", Q1_NAME, Q1_CYPHER, df_q1)
df_q1.head(20)

Q1: 0 elements have no property sets.
  Saved query -> queries/hvac/Q1_elements_without_psets.cypher
  Saved CSV   -> results/hvac/Q1_elements_without_psets.csv (0 rows)


""


## Q2 — Doors without a FireRating

In a pure HVAC discipline model there should be no doors. We expect 0 results — confirming the discipline separation worked.

In [3]:
Q2_NAME = "doors_missing_firerating"
Q2_CYPHER = """
MATCH (d:Element:IfcDoor)
OPTIONAL MATCH (d)-[:HAS_PSET]->(:PropertySet)-[:HAS_PROPERTY]->(p:Property {Name: 'FireRating'})
WITH d, p
WHERE p IS NULL OR p.IsEmpty = true
RETURN d.GlobalId AS GlobalId,
       d.Name     AS DoorName,
       d.Tag      AS Tag,
       CASE WHEN p IS NULL THEN 'Missing' ELSE 'Empty' END AS FireRatingStatus,
       p.Value    AS RawValue
ORDER BY FireRatingStatus, d.Name
"""
df_q2 = run_query(Q2_CYPHER)
print(f"Q2: {len(df_q2)} doors with missing/empty FireRating (HVAC model has no doors by design).")
save_query_and_results("Q2", Q2_NAME, Q2_CYPHER, df_q2)
df_q2

Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `IfcDoor` does not exist in database `hvac`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=18, offset=18>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 18, 'line': 2, 'column': 18}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (d:Element:IfcDoor)\nOPTIONAL MATCH (d)-[:HAS_PSET]->(:PropertySet)-[:HAS_PROPERTY]->(p:Property {Name: 'FireRating'})\nWITH d, p\nWHERE p IS NULL OR p.IsEmpty = true\nRETURN d.GlobalId AS GlobalId,\n       d.Name     AS DoorName,\n       d.Tag      AS Tag,\n       CASE WHEN p IS NULL THEN 'Missing' ELSE 'Empty' END 

Q2: 0 doors with missing/empty FireRating (HVAC model has no doors by design).
  Saved query -> queries/hvac/Q2_doors_missing_firerating.cypher
  Saved CSV   -> results/hvac/Q2_doors_missing_firerating.csv (0 rows)


""


## Q3 — Elements not assigned to any storey

In [4]:
Q3_NAME = "elements_without_storey"
Q3_CYPHER = """
MATCH (e:Element)
WHERE NOT EXISTS {
  MATCH (:Storey)-[:CONTAINS]->(e)
}
RETURN e.GlobalId AS GlobalId,
       e.IfcClass AS IfcClass,
       e.Name     AS Name,
       e.Tag      AS Tag
ORDER BY e.IfcClass
"""
df_q3 = run_query(Q3_CYPHER)
print(f"Q3: {len(df_q3)} elements not assigned to any storey.")
save_query_and_results("Q3", Q3_NAME, Q3_CYPHER, df_q3)
df_q3.head(20)

Q3: 2114 elements not assigned to any storey.
  Saved query -> queries/hvac/Q3_elements_without_storey.cypher
  Saved CSV   -> results/hvac/Q3_elements_without_storey.csv (2114 rows)


,GlobalId,IfcClass,Name,Tag
0,0rcx3NRlHCjxDJbhLB_yA3,IfcEnergyConversionDevice,M_Air Handling Unit - Split System - Horizonta...,621849
1,3eI1mbah1DKxvkhQWIGlDu,IfcEnergyConversionDevice,M_Air Handling Unit - Split System - Horizonta...,630076
2,1Hk4$wKjPFLAM7$8qJgCUa,IfcEnergyConversionDevice,M_Screw Chiller - Air Cooled - 281-1231 kW:633...,725669
3,1nnMo_AKzCgPoNKZdxVAD0,IfcFlowController,M_VAV Unit - Single Duct:200 mm:200 mm:720019,720019
4,0J3q8r36T0QQtYHes8eE2l,IfcFlowController,M_VAV Unit - Single Duct:150 mm:150 mm:710899,710899
5,37ERv$vjXCjPUDTL4UUcHs,IfcFlowController,M_VAV Unit - Single Duct:150 mm:150 mm:630300,630300
6,3E$HCkw6f2dvZkvEJvb0FE,IfcFlowController,M_VAV Unit - Single Duct:150 mm:150 mm:712077,712077
7,2eGk8b7S9EYgiKcuDXjVEb,IfcFlowController,M_VAV Unit - Single Duct:150 mm:150 mm:593948,593948
8,37ERv$vjXCjPUDTL4UUcU5,IfcFlowController,M_VAV Unit - Single Duct:150 mm:150 mm:630255,630255
9,2eGk8b7S9EYgiKcuDXjVFr,IfcFlowController,M_VAV Unit - Single Duct:150 mm:150 mm:593996,593996


## Q4 — Spaces with no name or number

In [5]:
Q4_NAME = "spaces_missing_identity"
Q4_CYPHER = """
MATCH (s:Space)
WHERE s.Name IS NULL     OR trim(s.Name) = ''
   OR s.LongName IS NULL OR trim(s.LongName) = ''
RETURN s.GlobalId                              AS GlobalId,
       coalesce(s.Name, '<MISSING>')           AS RoomNumber,
       coalesce(s.LongName, '<MISSING>')       AS RoomName,
       s.Description                           AS Description
ORDER BY s.GlobalId
"""
df_q4 = run_query(Q4_CYPHER)
print(f"Q4: {len(df_q4)} spaces missing name/number.")
save_query_and_results("Q4", Q4_NAME, Q4_CYPHER, df_q4)
df_q4.head(20)

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `Description` does not exist in database `hvac`. Verify that the spelling is correct.', position=<SummaryInputPosition line=8, column=10, offset=304>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 304, 'line': 8, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (s:Space)\nWHERE s.Name IS NULL     OR trim(s.Name) = ''\n   OR s.LongName IS NULL OR trim(s.LongName) = ''\nRETURN s.GlobalId                              AS GlobalId,\n       coalesce(s.Name, '<MISSING>')           AS RoomNumber,\n       coalesce(s.LongName, '<MISSING>')       AS RoomName,\n       s

Q4: 0 spaces missing name/number.
  Saved query -> queries/hvac/Q4_spaces_missing_identity.cypher
  Saved CSV   -> results/hvac/Q4_spaces_missing_identity.csv (0 rows)


""


## Q5 — Properties present but empty

In [6]:
Q5_NAME = "properties_with_empty_values"
Q5_CYPHER = """
MATCH (e:Element)-[:HAS_PSET]->(ps:PropertySet)-[:HAS_PROPERTY]->(p:Property)
WHERE p.IsEmpty = true
RETURN e.GlobalId AS ElementGlobalId,
       e.IfcClass AS IfcClass,
       e.Name     AS ElementName,
       ps.Name    AS PropertySet,
       p.Name     AS PropertyName,
       p.DataType AS DataType
ORDER BY e.IfcClass, ps.Name, p.Name
"""
df_q5 = run_query(Q5_CYPHER)
print(f"Q5: {len(df_q5)} property instances with empty values.")
save_query_and_results("Q5", Q5_NAME, Q5_CYPHER, df_q5)
df_q5.head(20)

Q5: 10729 property instances with empty values.
  Saved query -> queries/hvac/Q5_properties_with_empty_values.cypher
  Saved CSV   -> results/hvac/Q5_properties_with_empty_values.csv (10729 rows)


,ElementGlobalId,IfcClass,ElementName,PropertySet,PropertyName,DataType
0,0rcx3NRlHCjxDJbhLB_yA3,IfcEnergyConversionDevice,M_Air Handling Unit - Split System - Horizonta...,Electrical - Loads,Circuit Number,str
1,3eI1mbah1DKxvkhQWIGlDu,IfcEnergyConversionDevice,M_Air Handling Unit - Split System - Horizonta...,Electrical - Loads,Circuit Number,str
2,1Hk4$wKjPFLAM7$8qJgCUa,IfcEnergyConversionDevice,M_Screw Chiller - Air Cooled - 281-1231 kW:633...,Electrical - Loads,Circuit Number,str
3,0rcx3NRlHCjxDJbhLB_yA3,IfcEnergyConversionDevice,M_Air Handling Unit - Split System - Horizonta...,Electrical - Loads,Panel,str
4,3eI1mbah1DKxvkhQWIGlDu,IfcEnergyConversionDevice,M_Air Handling Unit - Split System - Horizonta...,Electrical - Loads,Panel,str
5,1Hk4$wKjPFLAM7$8qJgCUa,IfcEnergyConversionDevice,M_Screw Chiller - Air Cooled - 281-1231 kW:633...,Electrical - Loads,Panel,str
6,1nnMo_AKzCgPoNKZdxVAD0,IfcFlowController,M_VAV Unit - Single Duct:200 mm:200 mm:720019,Identity Data,Assembly Code,str
7,0J3q8r36T0QQtYHes8eE2l,IfcFlowController,M_VAV Unit - Single Duct:150 mm:150 mm:710899,Identity Data,Assembly Code,str
8,37ERv$vjXCjPUDTL4UUcHs,IfcFlowController,M_VAV Unit - Single Duct:150 mm:150 mm:630300,Identity Data,Assembly Code,str
9,3E$HCkw6f2dvZkvEJvb0FE,IfcFlowController,M_VAV Unit - Single Duct:150 mm:150 mm:712077,Identity Data,Assembly Code,str


## Q6 — Incompleteness ranked by IFC category

In [7]:
Q6_NAME = "incompleteness_by_category"
Q6_CYPHER = """
MATCH (e:Element)
OPTIONAL MATCH (e)-[:HAS_PSET]->(:PropertySet)-[:HAS_PROPERTY]->(p:Property)
WITH e.IfcClass AS Category, e, p
WITH Category,
     count(DISTINCT e) AS TotalElements,
     sum(CASE WHEN p IS NULL THEN 1 ELSE 0 END) AS ElementsNoProperties,
     count(p) AS TotalProperties,
     sum(CASE WHEN p.IsEmpty = true THEN 1 ELSE 0 END) AS EmptyProperties
RETURN Category, TotalElements, TotalProperties, EmptyProperties,
       CASE WHEN TotalProperties = 0 THEN 0.0
            ELSE round(toFloat(EmptyProperties) / TotalProperties, 3)
       END AS EmptyRatio
ORDER BY EmptyProperties DESC, EmptyRatio DESC
"""
df_q6 = run_query(Q6_CYPHER)
print(f"Q6: ranked {len(df_q6)} IFC categories by incompleteness.")
save_query_and_results("Q6", Q6_NAME, Q6_CYPHER, df_q6)
df_q6

Q6: ranked 6 IFC categories by incompleteness.
  Saved query -> queries/hvac/Q6_incompleteness_by_category.cypher
  Saved CSV   -> results/hvac/Q6_incompleteness_by_category.csv (6 rows)


,Category,TotalElements,TotalProperties,EmptyProperties,EmptyRatio
0,IfcFlowFitting,1590,61448,5275,0.086
1,IfcFlowSegment,1548,53253,4683,0.088
2,IfcFlowTerminal,440,26379,440,0.017
3,IfcFlowController,115,2914,309,0.106
4,IfcFlowMovingDevice,8,520,16,0.031
5,IfcEnergyConversionDevice,3,200,6,0.030


## Q7 — Duplicate identifiers and shared names

In [8]:
Q7A_CYPHER = """
MATCH (e:Element)
WITH e.GlobalId AS GlobalId, count(*) AS Occurrences, collect(e.IfcClass) AS Classes
WHERE Occurrences > 1
RETURN GlobalId, Occurrences, Classes
"""
Q7B_CYPHER = """
MATCH (e:Element)
WHERE e.Name IS NOT NULL AND trim(e.Name) <> ''
WITH e.IfcClass AS IfcClass, e.Name AS Name,
     count(*) AS Occurrences, collect(e.GlobalId)[..5] AS SampleGlobalIds
WHERE Occurrences > 1
RETURN IfcClass, Name, Occurrences, SampleGlobalIds
ORDER BY Occurrences DESC, IfcClass, Name
"""
df_q7a = run_query(Q7A_CYPHER)
df_q7b = run_query(Q7B_CYPHER)
print(f"Q7a: {len(df_q7a)} duplicate GlobalIds (should be 0).")
print(f"Q7b: {len(df_q7b)} Name+IfcClass combinations shared by multiple instances.")

(QUERIES_DIR / "Q7a_duplicate_globalids.cypher").write_text(Q7A_CYPHER.strip()+"\n", encoding="utf-8")
(QUERIES_DIR / "Q7b_duplicate_names.cypher").write_text(Q7B_CYPHER.strip()+"\n", encoding="utf-8")
df_q7a.to_csv(RESULTS_DIR / "Q7a_duplicate_globalids.csv", index=False)
df_q7b.to_csv(RESULTS_DIR / "Q7b_duplicate_names.csv", index=False)
print("  Saved Q7a + Q7b queries and CSVs.")
df_q7b.head(20)

Q7a: 0 duplicate GlobalIds (should be 0).
Q7b: 0 Name+IfcClass combinations shared by multiple instances.
  Saved Q7a + Q7b queries and CSVs.


""


## Q8 — Property values outside the permitted set

In [9]:
Q8_NAME = "values_outside_permitted_set"
Q8_CYPHER = """
MATCH (e:Element)-[:HAS_PSET]->(:PropertySet)-[:HAS_PROPERTY]->(p:Property)
WHERE p.IsEmpty = false
  AND (
    (p.Name = 'FireRating'
       AND NOT p.Value IN ['30','60','90','120','180','240',
                           'FD30','FD60','FD90','FD120','FD180','FD240'])
    OR
    (p.Name = 'IsExternal'
       AND NOT toLower(p.Value) IN ['true','false'])
  )
RETURN e.GlobalId AS ElementGlobalId,
       e.IfcClass AS IfcClass,
       e.Name     AS ElementName,
       p.Name     AS PropertyName,
       p.Value    AS RawValue,
       p.DataType AS DataType
ORDER BY p.Name, p.Value
"""
df_q8 = run_query(Q8_CYPHER)
print(f"Q8: {len(df_q8)} property values outside the permitted set.")
save_query_and_results("Q8", Q8_NAME, Q8_CYPHER, df_q8)
df_q8.head(20)

Q8: 0 property values outside the permitted set.
  Saved query -> queries/hvac/Q8_values_outside_permitted_set.cypher
  Saved CSV   -> results/hvac/Q8_values_outside_permitted_set.csv (0 rows)


""


## Q9 — Flow terminals without a flow-rate property (MEP-specific)

An HVAC air terminal that lacks a `NominalAirflowRate`, `AirflowRate`, or `FlowRate` property cannot be used in energy simulation, balancing calculations, or commissioning workflows. This is a Services-BIM-specific completeness check.

In [10]:
Q9_NAME = "flowterminals_missing_flowrate"
Q9_CYPHER = """
MATCH (e:Element:IfcFlowTerminal)
WHERE NOT EXISTS {
  MATCH (e)-[:HAS_PSET]->(:PropertySet)-[:HAS_PROPERTY]->(p:Property)
  WHERE p.IsEmpty = false
    AND (p.Name IN ['NominalAirflowRate','AirflowRate','FlowRate','NominalFlowRate'])
}
RETURN e.GlobalId AS GlobalId,
       e.IfcClass AS IfcClass,
       e.Name     AS Name,
       e.Tag      AS Tag
ORDER BY e.Name
"""
df_q9 = run_query(Q9_CYPHER)
print(f"Q9: {len(df_q9)} flow terminals without a populated flow rate property.")
save_query_and_results("Q9", Q9_NAME, Q9_CYPHER, df_q9)
df_q9.head(20)

Q9: 440 flow terminals without a populated flow rate property.
  Saved query -> queries/hvac/Q9_flowterminals_missing_flowrate.cypher
  Saved CSV   -> results/hvac/Q9_flowterminals_missing_flowrate.csv (440 rows)


,GlobalId,IfcClass,Name,Tag
0,31Wi8h84rAFhbjNsr8zAbg,IfcFlowTerminal,M_Exhaust Grill:ER-600 x 600 Face 300 x 300 Co...,585326
1,31Wi8h84rAFhbjNsr8zAbo,IfcFlowTerminal,M_Exhaust Grill:ER-600 x 600 Face 300 x 300 Co...,585334
2,26A3asOO536v6t$peFDNpZ,IfcFlowTerminal,M_Exhaust Grill:ER-600 x 600 Face 300 x 300 Co...,591017
3,2mEJjBcIX1YQdA7f2wXhSE,IfcFlowTerminal,M_Exhaust Grill:ER-600 x 600 Face 300 x 300 Co...,603451
4,2mEJjBcIX1YQdA7f2wXhTs,IfcFlowTerminal,M_Exhaust Grill:ER-600 x 600 Face 300 x 300 Co...,603459
5,2mEJjBcIX1YQdA7f2wXhT_,IfcFlowTerminal,M_Exhaust Grill:ER-600 x 600 Face 300 x 300 Co...,603467
6,2mEJjBcIX1YQdA7f2wXhTc,IfcFlowTerminal,M_Exhaust Grill:ER-600 x 600 Face 300 x 300 Co...,603475
7,1NAlaKy5fFXeFbPxNNJe2s,IfcFlowTerminal,M_Exhaust Grill:ER-600 x 600 Face 300 x 300 Co...,603760
8,1NAlaKy5fFXeFbPxNNJe2_,IfcFlowTerminal,M_Exhaust Grill:ER-600 x 600 Face 300 x 300 Co...,603768
9,1NAlaKy5fFXeFbPxNNJe16,IfcFlowTerminal,M_Exhaust Grill:ER-600 x 600 Face 300 x 300 Co...,603776


## Q10 — Elements not assigned to any IfcDistributionSystem (MEP-specific)

In a coordinated MEP model, flow elements should be grouped by their distribution system (supply air, return air, exhaust, chilled water, etc.). Elements with no system assignment cannot be properly identified for commissioning, FM handover, or system-based maintenance.

In [11]:
Q10_NAME = "elements_without_distribution_system"
# Since the HVAC model does not contain IfcDistributionSystem groupings,
# every flow element is unassigned — itself a data quality finding.
# The query below also confirms no DistributionSystem nodes exist.
Q10_CYPHER = """
MATCH (e:Element)
WHERE e.IfcClass IN ['IfcFlowSegment','IfcFlowFitting','IfcFlowTerminal',
                     'IfcFlowController','IfcFlowMovingDevice',
                     'IfcEnergyConversionDevice']
RETURN e.IfcClass AS IfcClass,
       count(*)   AS UnassignedCount
ORDER BY UnassignedCount DESC
"""
df_q10 = run_query(Q10_CYPHER)
print(f"Q10: Distribution-system assignment audit per flow class.")
print(f"     (No IfcDistributionSystem nodes exist in this model -> all flow elements are unassigned.)")
save_query_and_results("Q10", Q10_NAME, Q10_CYPHER, df_q10)
df_q10

Q10: Distribution-system assignment audit per flow class.
     (No IfcDistributionSystem nodes exist in this model -> all flow elements are unassigned.)
  Saved query -> queries/hvac/Q10_elements_without_distribution_system.cypher
  Saved CSV   -> results/hvac/Q10_elements_without_distribution_system.csv (6 rows)


,IfcClass,UnassignedCount
0,IfcFlowFitting,1590
1,IfcFlowSegment,1548
2,IfcFlowTerminal,440
3,IfcFlowController,115
4,IfcFlowMovingDevice,8
5,IfcEnergyConversionDevice,3


## Q11 — Duct/pipe segments missing nominal diameter (MEP-specific)

Flow segments without a populated `NominalDiameter`, `OuterDiameter`, or `Width/Height` property fail downstream pressure-drop, flow-balance, and bill-of-quantities workflows. Critical for Services BIM.

In [12]:
Q11_NAME = "segments_missing_diameter"
Q11_CYPHER = """
MATCH (e:Element)
WHERE e.IfcClass IN ['IfcFlowSegment','IfcFlowFitting']
  AND NOT EXISTS {
    MATCH (e)-[:HAS_PSET]->(:PropertySet)-[:HAS_PROPERTY]->(p:Property)
    WHERE p.IsEmpty = false
      AND p.Name IN ['NominalDiameter','OuterDiameter','InnerDiameter',
                     'NominalWidth','NominalHeight','Width','Height']
  }
RETURN e.IfcClass AS IfcClass,
       count(*)   AS SegmentsMissingDiameter
ORDER BY SegmentsMissingDiameter DESC
"""
df_q11 = run_query(Q11_CYPHER)
print(f"Q11: Flow segments / fittings missing dimensional properties (by class).")
save_query_and_results("Q11", Q11_NAME, Q11_CYPHER, df_q11)
df_q11

Q11: Flow segments / fittings missing dimensional properties (by class).
  Saved query -> queries/hvac/Q11_segments_missing_diameter.cypher
  Saved CSV   -> results/hvac/Q11_segments_missing_diameter.csv (2 rows)


,IfcClass,SegmentsMissingDiameter
0,IfcFlowFitting,1186
1,IfcFlowSegment,890


## Summary table (drop straight into README §4.1)

In [13]:
summary = pd.DataFrame([
    {"Question": "Q1",  "Dimension": "Completeness",          "Issue": "Elements with no property sets",       "HVAC Findings": len(df_q1)},
    {"Question": "Q2",  "Dimension": "Completeness (compliance)", "Issue": "Doors missing FireRating",          "HVAC Findings": len(df_q2)},
    {"Question": "Q3",  "Dimension": "Consistency (spatial)",  "Issue": "Elements not in any storey",           "HVAC Findings": len(df_q3)},
    {"Question": "Q4",  "Dimension": "Completeness (ID)",      "Issue": "Spaces missing name/number",           "HVAC Findings": len(df_q4)},
    {"Question": "Q5",  "Dimension": "Completeness",           "Issue": "Properties with empty values",         "HVAC Findings": len(df_q5)},
    {"Question": "Q6",  "Dimension": "Completeness (agg.)",    "Issue": "Incompleteness ranked by category",    "HVAC Findings": f"{len(df_q6)} categories ranked"},
    {"Question": "Q7",  "Dimension": "Uniqueness",             "Issue": "Duplicate GlobalIds | shared Names",   "HVAC Findings": f"{len(df_q7a)} | {len(df_q7b)}"},
    {"Question": "Q8",  "Dimension": "Validity",               "Issue": "Values outside permitted set",         "HVAC Findings": len(df_q8)},
    {"Question": "Q9",  "Dimension": "Completeness (MEP)",     "Issue": "Flow terminals missing flow rate",     "HVAC Findings": len(df_q9)},
    {"Question": "Q10", "Dimension": "Consistency (MEP)",      "Issue": "Flow elements without DistributionSystem", "HVAC Findings": f"{len(df_q10)} classes (no systems defined)"},
    {"Question": "Q11", "Dimension": "Completeness (MEP)",     "Issue": "Flow segments missing diameter",       "HVAC Findings": f"{len(df_q11)} classes ranked"},
])
summary.to_csv(RESULTS_DIR / "00_summary.csv", index=False)
print("Saved results/hvac/00_summary.csv")
summary

Saved results/hvac/00_summary.csv


,Question,Dimension,Issue,HVAC Findings
0,Q1,Completeness,Elements with no property sets,0
1,Q2,Completeness (compliance),Doors missing FireRating,0
2,Q3,Consistency (spatial),Elements not in any storey,2114
3,Q4,Completeness (ID),Spaces missing name/number,0
4,Q5,Completeness,Properties with empty values,10729
5,Q6,Completeness (agg.),Incompleteness ranked by category,6 categories ranked
6,Q7,Uniqueness,Duplicate GlobalIds | shared Names,0 | 0
7,Q8,Validity,Values outside permitted set,0
8,Q9,Completeness (MEP),Flow terminals missing flow rate,440
9,Q10,Consistency (MEP),Flow elements without DistributionSystem,6 classes (no systems defined)


In [14]:
driver.close()
print("Driver closed. All queries complete.")

Driver closed. All queries complete.
